In [46]:
from random import randbytes

# Utils
def xor_data(x, y):
    return bytes([a ^^ b for a, b in zip(x, y)])

def get_bit_from_bytes(array: bytes, i: int):
    return (array[i // 8] >> (7 - i % 8)) & 1
    

In [47]:
class Trivium:

    def __init__(self, key: bytes, iv: bytes):
        if len(key) != 10 or len(iv) != 10:
            raise ValueError("Key and IV must each be 10 bytes long.")
            
        # Initialization
        self.key = key
        self.iv = iv
        self.state = [0] * 288
        
        # (s0, s1, ..., s92) ← (K0, K1, ..., K79, 0, ..., 0)
        for i in range(80):
            self.state[i] = get_bit_from_bytes(key, i)
        
        # (s93, s94, ..., s176) ← (IV0, IV1, ..., IV79, 0, ..., 0)
        for i in range(80):
            self.state[93 + i] = get_bit_from_bytes(iv, i)
            
        # (s177, s278, ..., s287) ← (0, ..., 0, 1, 1, 1)
        self.state[285] = self.state[286] = self.state[287] = 1
        
        self.reset()
    
    def keystream(self, n: int):
        return [self.keystream_step() for _ in range(n)]
    
    def keystream_step(self):
        # t1 ← s65 + s92
        t1 = self.state[65] ^^ self.state[92]
        
        # t2 ← s161 + s176
        t2 = self.state[161] ^^ self.state[176]
        
        # t3 ← s242 + s287
        t3 = self.state[242] ^^ self.state[287]
        
        z = t1 ^^ t2 ^^ t3
        
        # t1 ← t1 + s90 * s91 + s170
        t1 = t1 ^^ (self.state[90] & self.state[91]) ^^ self.state[170]
        
        # t2 ← t2 + s174 * s175 + s263
        t2 = t2 ^^ (self.state[174] & self.state[175]) ^^ self.state[263]
        
        # t3 ← t3 + s285 * s286 + s68
        t3 = t3 ^^ (self.state[285] & self.state[286]) ^^ self.state[68]
        
        # (s0, s1, ..., s92) ← (t3, s0, ..., s91)
        # (s93, s94, ..., s176) ← (t1, s93, ..., s175)
        # (s177, s278, ..., s287) ← (t2, s177, ..., s286)
        self.state = [t3] + self.state[0:92] + [t1] + self.state[93:176] + [t2] + self.state[177:287]
        
        return z
    
    def reset(self):
        self.keystream(4 * 288) # ignore result

In [48]:
class TrivAlg:
    
    def __init__(self, key: bytes, iv: bytes):
        self.key = key
        self.iv = iv

    def encrypt(self, plaintext: bytes):
        triv = Trivium(self.key, self.iv)
        keystream = triv.keystream(len(plaintext))
        return xor_data(plaintext, keystream)

    def decrypt(self, ciphertext: bytes):
        return self.encrypt(ciphertext)

In [51]:
#key: bytes = randbytes(10)
key: bytes = b"zzzzzzzzzz"
#iv: bytes = randbytes(10)
iv: bytes = b"1234567890"

triv = TrivAlg(key, iv)

plaintext: bytes = b"Hello, Trivium!"
cipher: bytes = triv.encrypt(plaintext)
decrypted: bytes = triv.decrypt(cipher)

print("Plaintext:", plaintext)
print("Ciphertext:", cipher)
print("Decrypted text:", decrypted)
assert plaintext == decrypted

Plaintext: b'Hello, Trivium!'
Ciphertext: b'Iemlo- Ushwitl!'
Decrypted text: b'Hello, Trivium!'
